<a href="https://colab.research.google.com/github/maruson08/new-folder-3/blob/main/%EB%B0%95%ED%8E%B8_%EC%86%90%EC%83%81%EB%A5%A0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install opencv-python

import cv2
import numpy as np
import os
from google.colab import files
import matplotlib.pyplot as plt

In [ ]:
# 이미지 파일 업로드
uploaded = files.upload()

# 원본 파일 이름 /content/~~.jpg로 수정
ORIGINAL_IMAGE_FILE = 'image_fd8af9.png'

uploaded_files = list(uploaded.keys())
print(f"\n업로드된 파일 목록: {uploaded_files}")

if ORIGINAL_IMAGE_FILE in uploaded_files:
    uploaded_files.remove(ORIGINAL_IMAGE_FILE)
    print(f"✅ 원본 이미지 파일 확인: {ORIGINAL_IMAGE_FILE}")
elif uploaded_files:
    ORIGINAL_IMAGE_FILE = uploaded_files.pop(0)
    print(f"⚠️ 경고: '{ORIGINAL_IMAGE_FILE}'을 원본 이미지로 임시 설정합니다. 이름을 확인해 주세요.")
else:
    raise FileNotFoundError("업로드된 파일이 없습니다.")

print(f"\n손상된 이미지로 처리할 파일 목록: {uploaded_files}")

Saving Screenshot 2025-11-05 100307.png to Screenshot 2025-11-05 100307.png
Saving Screenshot 2025-11-05 100300.png to Screenshot 2025-11-05 100300 (1).png

업로드된 파일 목록: ['Screenshot 2025-11-05 100307.png', 'Screenshot 2025-11-05 100300 (1).png']
⚠️ 경고: 'Screenshot 2025-11-05 100307.png'을 원본 이미지로 임시 설정합니다. 정확한 이름을 입력해 주세요.

손상된 이미지(damaged_photos)로 이동할 파일:
- Screenshot 2025-11-05 100300 (1).png


In [ ]:
def calculate_damage_rate_with_thresholding(original_img_name, damaged_file_list):
    """
    Colab 환경에서 업로드된 파일을 사용하여 손상률을 계산합니다.
    - 회색조 이미지를 이진화(Otsu)하여 처리합니다.
    - 크기가 다를 경우 원본 크기에 맞게 자동 조정합니다.
    """

    print("\n" + "=" * 50)
    print("--- 🔬 박편 손상률 분석 시작 (이진화 및 크기 조정 적용) ---")

    original_img = cv2.imread(original_img_name, cv2.IMREAD_GRAYSCALE)

    if original_img is None:
        print(f"오류: 원본 이미지 파일({original_img_name})을 찾거나 읽을 수 없습니다.")
        return

    _, original_thresh = cv2.threshold(original_img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    roi_pixels_count = np.sum(original_thresh == 0)

    if roi_pixels_count == 0:
        print("경고: 이진화 후 원본 이미지에서 관심 영역(검은색 픽셀)을 찾을 수 없습니다. 손상률 계산 불가.")
        return

    original_height, original_width = original_img.shape[:2]

    results = []

    for filename in damaged_file_list:

        damaged_img = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)

        if damaged_img is None:
            print(f"오류: {filename} 파일을 읽을 수 없습니다. 건너뜀.")
            continue

        if damaged_img.shape[:2] != original_img.shape[:2]:
            print(f"**크기 조정:** {filename} 크기({damaged_img.shape[:2]})가 원본({original_img.shape[:2]})과 달라 조정합니다.")
            damaged_img = cv2.resize(damaged_img, (original_width, original_height), interpolation=cv2.INTER_AREA)

        _, damaged_thresh = cv2.threshold(damaged_img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

        original_black_area = (original_thresh == 0)

        damaged_white_area = (damaged_thresh == 255)

        damage_pixels = np.logical_and(original_black_area, damaged_white_area)

        damage_pixels_count = np.sum(damage_pixels)

        damage_rate = (damage_pixels_count / roi_pixels_count) * 100

        results.append({
            'file_name': filename,
            'damage_rate': damage_rate,
            'damage_pixels': damage_pixels_count,
            'roi_pixels': roi_pixels_count
        })

    print("-" * 50)
    print(f"**원본 박편 픽셀 수 (ROI): {roi_pixels_count}개**")
    print("-" * 50)

    print("## ✨ 최종 손상률 계산 결과")
    for res in results:
        print(f"| **파일 이름**: {res['file_name']:<20} | **손상률**: {res['damage_rate']:.2f}% |")

    print("=" * 50)

    return original_thresh, results

In [ ]:
original_thresh, results = calculate_damage_rate_with_thresholding(ORIGINAL_IMAGE_FILE, uploaded_files)

--- 🔬 박편 손상률 분석 시작 ---
경고: Screenshot 2025-11-05 100300 (1).png의 크기((340, 1007))가 원본((339, 1009))과 다릅니다. 건너뜁니다.
--------------------------------------------------
**원본 박편 픽셀 수 (ROI): 102771개**
--------------------------------------------------
## ✨ 최종 손상률 계산 결과
--------------------------------------------------


# 시각화(옵션)

In [ ]:
if original_thresh is not None and results:

    print("\n## 🖼️ 손상 영역 시각화 (첫 번째 손상된 이미지)")

    first_result = results[0]
    first_filename = first_result['file_name']

    damaged_img = cv2.imread(first_filename, cv2.IMREAD_GRAYSCALE)
    if damaged_img.shape[:2] != original_thresh.shape[:2]:
        height, width = original_thresh.shape[:2]
        damaged_img = cv2.resize(damaged_img, (width, height), interpolation=cv2.INTER_AREA)

    _, damaged_thresh = cv2.threshold(damaged_img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    original_black_area = (original_thresh == 0)
    damaged_white_area = (damaged_thresh == 255)
    damage_mask = np.logical_and(original_black_area, damaged_white_area)

    damage_overlay = np.zeros((*original_thresh.shape, 3), dtype=np.uint8)
    damage_overlay[damage_mask] = [255, 0, 0]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(original_thresh, cmap='gray')
    axes[0].set_title(f"1. 이진화된 원본 ({ORIGINAL_IMAGE_FILE})")
    axes[0].axis('off')

    axes[1].imshow(damaged_thresh, cmap='gray')
    axes[1].set_title(f"2. 이진화된 손상 ({first_filename})\n손상률: {first_result['damage_rate']:.2f}%")
    axes[1].axis('off')

    original_color = cv2.cvtColor(original_thresh, cv2.COLOR_GRAY2BGR)
    final_overlay = cv2.addWeighted(original_color, 1.0, damage_overlay, 0.5, 0)

    axes[2].imshow(cv2.cvtColor(final_overlay, cv2.COLOR_BGR2RGB))
    axes[2].set_title("3. 계산된 손상 영역 (빨간색)")
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()